# 🎙️ Conformer U-Net: Speech Enhancement Training in Google Colab
This notebook trains the **Conformer U-Net** on the **VoiceBank-DEMAND** dataset using a free GPU runtime in Google Colab.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

In [ ]:
# 2. Install Dependencies
!pip install -q datasets soundfile pesq pystoi matplotlib

In [ ]:
# 3. Fast & Reliable Dataset Preparation (via Hugging Face)
import os, soundfile as sf
from datasets import load_dataset

print("📥 Fetching VoiceBank-DEMAND dataset...")
# Downloads train and test splits directly
train_ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k", split="train[:2000]")
val_ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k", split="test[:200]")

os.makedirs("data/clean_train", exist_ok=True)
os.makedirs("data/noisy_train", exist_ok=True)
os.makedirs("data/clean_val", exist_ok=True)
os.makedirs("data/noisy_val", exist_ok=True)

print("💾 Saving training audio files...")
for i, item in enumerate(train_ds):
    sf.write(f"data/clean_train/{i:05d}.wav", item["clean"]["array"], 16000)
    sf.write(f"data/noisy_train/{i:05d}.wav", item["noisy"]["array"], 16000)

print("💾 Saving validation audio files...")
for i, item in enumerate(val_ds):
    sf.write(f"data/clean_val/{i:05d}.wav", item["clean"]["array"], 16000)
    sf.write(f"data/noisy_val/{i:05d}.wav", item["noisy"]["array"], 16000)

print(f"✅ Dataset ready! Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

In [ ]:
# 4. Launch Training Loop with Mixed Precision (AMP)
!python train.py \
    --clean_train_dir ./data/clean_train \
    --noisy_train_dir ./data/noisy_train \
    --clean_val_dir ./data/clean_val \
    --noisy_val_dir ./data/noisy_val \
    --epochs 20 \
    --batch_size 16 \
    --lr 0.0005 \
    --save_dir ./checkpoints

In [ ]:
# 5. Run Quantitative Evaluation & Benchmarking
!python evaluate.py \
    --checkpoint ./checkpoints/best_model.pth \
    --clean_test_dir ./data/clean_val \
    --noisy_test_dir ./data/noisy_val

In [ ]:
# 6. Launch Live Gradio Demo
!python app.py